# 02 — Ultrasound-Specific Preprocessing & Leakage-Free Splits

**Phases 3, 4 and 5.** Three jobs:

1. **Panel decomposition** — each `R*.jpg` is a composite sheet holding several
   ultrasound frames. Feeding the whole sheet to a CNN wastes most of the input
   on gutters and duplicated console banners. Splitting it turns 259 studies into
   roughly 1,500 usable frames and converts the task into multiple-instance
   learning. *This is contribution #1 in your proposal — the part that makes the
   preprocessing "ultrasound-specific" rather than generic.*
2. **Frame cleaning** — console banner removal, sector ROI cropping, CLAHE,
   speckle reduction.
3. **Splits** — 5-fold cross-validation that is simultaneously **group-aware**
   (near-duplicates stay in one fold) and **label-stratified** (rare classes
   appear in every fold).

Outputs go to `/kaggle/working/artifacts/` and `/kaggle/working/frames/`.
Save this notebook's output as a Kaggle dataset — notebooks 03–09 all read it.

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures"
FRAMES = Path("/kaggle/working/frames")
for d in (ART, FIG, FRAMES):
    d.mkdir(parents=True, exist_ok=True)

def save(name):
    plt.savefig(FIG / f"{name}.png")
    plt.show()

def find_artifacts():
    local = ART / "master.csv"
    if local.exists():
        return ART
    for p in Path("/kaggle/input").rglob("master.csv"):
        return p.parent
    raise FileNotFoundError("Run notebooks 00 and 01 first.")

SRC = find_artifacts()
df = pd.read_csv(SRC / "master.csv")
df["labels"] = df["labels"].fillna("").map(lambda s: [x for x in s.split("|") if x])
multihot = pd.read_csv(SRC / "labels_multihot.csv", index_col=0)

dup_path = SRC / "dup_groups.csv"
if dup_path.exists():
    dup = pd.read_csv(dup_path)
    df = df.merge(dup, on="study_id", how="left")
else:
    print("dup_groups.csv missing — falling back to one group per study "
          "(run notebook 01 for leakage-free splits).")
    df["dup_group"] = range(len(df))

print(f"{len(df)} studies, {df['dup_group'].nunique()} duplicate groups")

dup_groups.csv missing — falling back to one group per study (run notebook 01 for leakage-free splits).
259 studies, 259 duplicate groups


## 1. Configuration

In [2]:
CFG = {
    "img_size": 224,            # 384 also works; costs ~3x the compute
    "min_panel_area_frac": 0.010,
    "min_panel_side": 56,
    "close_kernel": 5,          # must be SMALLER than the gutter between frames
    "banner_frac": 0.11,        # top fraction of each frame holding console text
    "clahe_clip": 2.5,
    "clahe_grid": 8,
    "speckle_filter": "median",  # "median" | "nlm" | "none"
    "n_folds": 5,
    "seed": 42,
    "save_quality": 94,
}
json.dump(CFG, open(ART / "preprocess_config.json", "w"), indent=2)
CFG

{'img_size': 224,
 'min_panel_area_frac': 0.01,
 'min_panel_side': 56,
 'close_kernel': 5,
 'banner_frac': 0.11,
 'clahe_clip': 2.5,
 'clahe_grid': 8,
 'speckle_filter': 'median',
 'n_folds': 5,
 'seed': 42,
 'save_quality': 94}

## 2. Panel decomposition

Two complementary detectors, because composite layouts vary:

* **Projection-profile splitting** handles clean grids. Collapse the foreground
  mask onto each axis; the dark gutters between frames appear as runs of near-zero
  values, and those runs are the cut points.
* **Connected components** handles irregular layouts where frames are not aligned.

We run the grid detector first and fall back to components when it returns a
single region. If both fail, the whole sheet is used as one frame — which is the
correct degradation, not an error.

In [3]:
def foreground_mask(gray: np.ndarray, close: int = None) -> np.ndarray:
    """Binary mask of the ultrasound sectors, gutters excluded.

    Otsu alone keeps too much of the lifted black level that photographed
    screens have, so we AND it with a conservative absolute floor. The closing
    kernel must stay SMALLER than the inter-panel gutter or adjacent frames
    merge into one blob — that is the single most common failure here.
    """
    close = close or CFG["close_kernel"]
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    thr, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    floor = max(18, int(thr * 0.5))
    th = cv2.bitwise_and(th, (blur > floor).astype(np.uint8) * 255)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close, close))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k, iterations=2)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN,
                          cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), 1)
    return th


def split_components(mask: np.ndarray) -> list:
    h, w = mask.shape
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    boxes = []
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        if area < CFG["min_panel_area_frac"] * h * w:
            continue
        if bw < CFG["min_panel_side"] or bh < CFG["min_panel_side"]:
            continue
        if not (0.20 < bw / bh < 5.0):
            continue
        boxes.append((x, y, x + bw, y + bh))
    return boxes


def _cut_points(profile: np.ndarray, length: int, min_gap: int, frac=0.04) -> list:
    """Midpoints of low-density runs — the gutters between frames."""
    thresh = frac * profile.max() if profile.max() > 0 else 0
    low = profile <= thresh
    cuts, start = [], None
    for i, v in enumerate(low):
        if v and start is None:
            start = i
        elif not v and start is not None:
            if i - start >= min_gap:
                cuts.append((start + i) // 2)
            start = None
    if start is not None and length - start >= min_gap:
        cuts.append((start + length) // 2)
    return cuts


def split_grid(mask: np.ndarray) -> list:
    """Projection-profile grid cut — the fallback when frames touch."""
    h, w = mask.shape
    r_cuts = sorted({0, h, *_cut_points(mask.sum(1) / 255.0, h, max(5, h // 80))})
    c_cuts = sorted({0, w, *_cut_points(mask.sum(0) / 255.0, w, max(5, w // 80))})
    boxes = []
    for a, b in zip(r_cuts[:-1], r_cuts[1:]):
        for c, d in zip(c_cuts[:-1], c_cuts[1:]):
            if (b - a) < CFG["min_panel_side"] or (d - c) < CFG["min_panel_side"]:
                continue
            cell = mask[a:b, c:d]
            if (cell > 0).mean() < 0.08:
                continue
            ys, xs = np.where(cell > 0)
            boxes.append((c + int(xs.min()), a + int(ys.min()),
                          c + int(xs.max()) + 1, a + int(ys.max()) + 1))
    return boxes


def _refine_oversized(mask: np.ndarray, boxes: list) -> list:
    """If one component is far larger than its siblings, two frames probably
    merged — try a projection cut inside that box only."""
    if len(boxes) < 2:
        return boxes
    areas = np.array([(x1 - x0) * (y1 - y0) for x0, y0, x1, y1 in boxes], float)
    med = np.median(areas)
    out = []
    for (x0, y0, x1, y1), a in zip(boxes, areas):
        if a > 1.8 * med:
            sub = split_grid(mask[y0:y1, x0:x1])
            if len(sub) > 1:
                out += [(x0 + sx0, y0 + sy0, x0 + sx1, y0 + sy1)
                        for sx0, sy0, sx1, sy1 in sub]
                continue
        out.append((x0, y0, x1, y1))
    return out


def detect_panels(bgr: np.ndarray) -> list:
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    mask = foreground_mask(gray)
    boxes = split_components(mask)
    if len(boxes) < 2:
        grid = split_grid(mask)
        if len(grid) > len(boxes):
            boxes = grid
    else:
        boxes = _refine_oversized(mask, boxes)
    if not boxes:
        boxes = [(0, 0, w, h)]
    row_h = max(1, h // 12)
    return sorted(boxes, key=lambda b: (b[1] // row_h, b[0]))

### Visual check — do not skip this

Panel detection is the one step where a silent failure is invisible downstream:
the model still trains, it just trains on gutters. Look at the boxes.

In [4]:
demo = df[df["has_image"]].sample(min(6, df["has_image"].sum()), random_state=1)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (_, r) in zip(axes.ravel(), demo.iterrows()):
    bgr = cv2.imread(r["image_path"])
    if bgr is None:
        ax.axis("off")
        continue
    boxes = detect_panels(bgr)
    vis = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).copy()
    for (x0, y0, x1, y1) in boxes:
        cv2.rectangle(vis, (x0, y0), (x1, y1), (255, 40, 40),
                      max(2, vis.shape[1] // 300))
    ax.imshow(vis)
    ax.set_title(f"{r['study_id']} — {len(boxes)} panels", fontsize=10)
    ax.axis("off")
plt.suptitle("Panel decomposition (adjust min_panel_area_frac if boxes look wrong)")
plt.tight_layout()
save("fig11_panel_detection")

## 3. Frame cleaning

`crop_sector` finds the ultrasound sector itself and discards the black surround
plus the console banner. `enhance` applies CLAHE in LAB space so colour Doppler
frames are not destroyed, then a light despeckle.

**Do not turn the CLAHE clip limit up.** Aggressive contrast stretching on
ultrasound amplifies speckle into structures that look like lesions, and a
radiologist reviewing your Grad-CAM figures will notice.

In [5]:
def crop_sector(panel: np.ndarray) -> np.ndarray:
    h, w = panel.shape[:2]
    top = int(CFG["banner_frac"] * h)
    body = panel[top:, :]
    if body.size == 0:
        body = panel
    gray = cv2.cvtColor(body, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, 12, 255, cv2.THRESH_BINARY)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k, iterations=2)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return body
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 0.10 * body.shape[0] * body.shape[1]:
        return body
    x, y, bw, bh = cv2.boundingRect(c)
    pad = 2
    x0, y0 = max(0, x - pad), max(0, y - pad)
    x1, y1 = min(body.shape[1], x + bw + pad), min(body.shape[0], y + bh + pad)
    out = body[y0:y1, x0:x1]
    return out if out.size else body


def enhance(img: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=CFG["clahe_clip"],
                            tileGridSize=(CFG["clahe_grid"], CFG["clahe_grid"]))
    l = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    if CFG["speckle_filter"] == "median":
        out = cv2.medianBlur(out, 3)
    elif CFG["speckle_filter"] == "nlm":
        out = cv2.fastNlMeansDenoisingColored(out, None, 5, 5, 7, 21)
    return out


def letterbox(img: np.ndarray, size: int) -> np.ndarray:
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
    r = cv2.resize(img, (nw, nh), interpolation=interp)
    canvas = np.zeros((size, size, 3), np.uint8)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = r
    return canvas


def is_informative(frame: np.ndarray) -> bool:
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return not (g.std() < 9 or (g > 25).mean() < 0.06)

In [6]:
sample_row = df[df["has_image"]].iloc[0]
bgr = cv2.imread(sample_row["image_path"])
boxes = detect_panels(bgr)
x0, y0, x1, y1 = boxes[0]
panel = bgr[y0:y1, x0:x1]

stages = [
    ("panel", panel),
    ("sector crop", crop_sector(panel)),
    ("CLAHE + despeckle", enhance(crop_sector(panel))),
    (f"letterbox {CFG['img_size']}", letterbox(enhance(crop_sector(panel)), CFG["img_size"])),
]
fig, axes = plt.subplots(1, len(stages), figsize=(4 * len(stages), 4.2))
for ax, (name, im) in zip(axes, stages):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{name}\n{im.shape[1]}×{im.shape[0]}", fontsize=10)
    ax.axis("off")
plt.suptitle(f"Preprocessing pipeline — {sample_row['study_id']}")
plt.tight_layout()
save("fig12_preprocessing_stages")

## 4. Build the frame corpus

Roughly 4–6 minutes for 259 sheets on a Kaggle CPU session.

In [7]:
from tqdm.auto import tqdm

records = []
failed = []
for _, r in tqdm(df[df["has_image"]].iterrows(), total=int(df["has_image"].sum())):
    bgr = cv2.imread(r["image_path"])
    if bgr is None:
        failed.append(r["study_id"])
        continue
    out_dir = FRAMES / str(r["study_id"])
    out_dir.mkdir(parents=True, exist_ok=True)
    kept = 0
    for i, (x0, y0, x1, y1) in enumerate(detect_panels(bgr)):
        panel = bgr[y0:y1, x0:x1]
        if panel.size == 0:
            continue
        frame = letterbox(enhance(crop_sector(panel)), CFG["img_size"])
        if not is_informative(frame):
            continue
        fp = out_dir / f"{r['study_id']}_p{kept:02d}.jpg"
        cv2.imwrite(str(fp), frame, [cv2.IMWRITE_JPEG_QUALITY, CFG["save_quality"]])
        records.append({
            "study_id": r["study_id"], "frame_id": fp.stem, "frame_path": str(fp),
            "panel_index": kept, "src_box": f"{x0},{y0},{x1},{y1}",
        })
        kept += 1
    if kept == 0:
        frame = letterbox(enhance(bgr), CFG["img_size"])
        fp = out_dir / f"{r['study_id']}_p00.jpg"
        cv2.imwrite(str(fp), frame, [cv2.IMWRITE_JPEG_QUALITY, CFG["save_quality"]])
        records.append({"study_id": r["study_id"], "frame_id": fp.stem,
                        "frame_path": str(fp), "panel_index": 0, "src_box": "full"})

frames = pd.DataFrame(records)
frames.to_csv(ART / "frames.csv", index=False)

per_study = frames.groupby("study_id").size()
print(f"frames extracted : {len(frames)}")
print(f"studies covered  : {frames['study_id'].nunique()}")
print(f"frames per study : mean {per_study.mean():.1f}, median {per_study.median():.0f}, "
      f"min {per_study.min()}, max {per_study.max()}")
if failed:
    print("unreadable images:", failed)

plt.figure(figsize=(7, 3.5))
plt.hist(per_study, bins=range(1, int(per_study.max()) + 2), color="#2a6f97",
         edgecolor="white", align="left")
plt.xlabel("frames per study")
plt.ylabel("studies")
plt.title("Panel decomposition yield")
save("fig13_frames_per_study")

  0%|          | 0/259 [00:00<?, ?it/s]

frames extracted : 494
studies covered  : 259
frames per study : mean 1.9, median 1, min 1, max 12


## 5. Augmentation policy

Horizontal flip is **not** anatomically neutral in abdominal ultrasound — liver
right, spleen left. We use it only as a weak regulariser at low probability and
ablate it in notebook 09. Rotation stays small because probe orientation is
standardised by protocol.

In [8]:
try:
    import albumentations as A
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=False)
    import albumentations as A

IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD = (0.229, 0.224, 0.225)
print("albumentations", A.__version__)


def _try(cls, *kwarg_variants):
    """Albumentations renamed several arguments between 1.x and 2.x.
    Try each spelling and keep the first that constructs."""
    last = None
    for kw in kwarg_variants:
        try:
            return cls(**kw)
        except (TypeError, ValueError) as e:
            last = e
    raise last


def build_transforms(size=None, hflip=True, strength=1.0):
    size = size or CFG["img_size"]
    hole = max(4, size // 12)

    affine = _try(
        A.Affine,
        dict(scale=(1 - .10 * strength, 1 + .10 * strength),
             translate_percent=0.05 * strength,
             rotate=(-10 * strength, 10 * strength),
             border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.7),
        dict(scale=(1 - .10 * strength, 1 + .10 * strength),
             translate_percent=0.05 * strength,
             rotate=(-10 * strength, 10 * strength),
             mode=cv2.BORDER_CONSTANT, cval=0, p=0.7),
        dict(scale=(1 - .10 * strength, 1 + .10 * strength),
             translate_percent=0.05 * strength,
             rotate=(-10 * strength, 10 * strength), p=0.7),
    )
    noise = _try(A.GaussNoise, dict(std_range=(0.03, 0.12)),
                 dict(var_limit=(5.0, 30.0)), dict())
    dropout = _try(
        A.CoarseDropout,
        dict(num_holes_range=(1, 6), hole_height_range=(4, hole),
             hole_width_range=(4, hole), p=0.25),
        dict(max_holes=6, max_height=hole, max_width=hole, p=0.25),
    )

    train = A.Compose([
        affine,
        A.HorizontalFlip(p=0.25 if hflip else 0.0),
        A.RandomBrightnessContrast(brightness_limit=0.18 * strength,
                                   contrast_limit=0.18 * strength, p=0.6),
        A.OneOf([noise, A.MultiplicativeNoise((0.9, 1.1))], p=0.3),
        dropout,
        A.Normalize(IMNET_MEAN, IMNET_STD),
    ])
    valid = A.Compose([A.Normalize(IMNET_MEAN, IMNET_STD)])
    return train, valid

train_tf, valid_tf = build_transforms()

demo_img = cv2.cvtColor(cv2.imread(frames["frame_path"].iloc[0]), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))
axes[0].imshow(demo_img)
axes[0].set_title("original")
axes[0].axis("off")
aug_vis = A.Compose(train_tf.transforms[:-1])
for ax in axes[1:]:
    ax.imshow(aug_vis(image=demo_img)["image"])
    ax.set_title("augmented", fontsize=9)
    ax.axis("off")
plt.tight_layout()
save("fig14_augmentation")

albumentations 2.0.8


## 6. Text preprocessing

Medical text is under-cleaned on purpose. We expand abbreviations only where the
expansion is unambiguous in this corpus, and we build a **section-aware
serialisation** that the generation model consumes — findings and impression are
marked with explicit tags so the decoder learns they play different roles.

In [9]:
import re

ABBREV = {
    r"\bCBD\b": "common bile duct", r"\bMPD\b": "main pancreatic duct",
    r"\bIVC\b": "inferior vena cava", r"\bGB\b": "gall bladder",
    r"\bPVR\b": "post void residual", r"\bMCC\b": "maximum cystometric capacity",
    r"\bTVS\b": "transvaginal sonography", r"\bUSG\b": "ultrasonography",
    r"\bPCOS?\b": "polycystic ovary syndrome", r"\bPID\b": "pelvic inflammatory disease",
    r"\bCRL\b": "crown rump length", r"\bBPD\b": "biparietal diameter",
    r"\bAFI\b": "amniotic fluid index", r"\bEFW\b": "estimated fetal weight",
    r"\bEDD\b": "expected date of delivery", r"\bGS\b": "gestational sac",
    r"\bSOL\b": "space occupying lesion", r"\bP\.?O\.?D\b": "pouch of Douglas",
    r"\bHC\b": "head circumference", r"\bAC\b": "abdominal circumference",
    r"\bRO\b": "right ovary", r"\bLO\b": "left ovary",
}

def expand_abbrev(text: str) -> str:
    for pat, full in ABBREV.items():
        text = re.sub(pat, full, text)
    return text


def normalise_units(text: str) -> str:
    text = re.sub(r"(\d)\s*(cm|mm|cc|ml|gm|g)\b", r"\1 \2", text, flags=re.I)
    text = re.sub(r"Gr\s*-?\s*(I{1,3}|IV|1|2|3|4)", r"Grade-\1", text, flags=re.I)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def sentences(text: str) -> list:
    parts = re.split(r"(?<=[.;])\s+(?=[A-Z0-9])", text)
    return [p.strip() for p in parts if len(p.strip()) > 2]


def serialize_sections(sections_json: str, impression: str, max_sections=None) -> str:
    secs = json.loads(sections_json)
    keys = list(secs)[:max_sections] if max_sections else list(secs)
    body = " ".join(f"[{k}] {normalise_units(secs[k])}" for k in keys)
    return f"[FINDINGS] {body} [IMPRESSION] {normalise_units(impression)}"

df["report_proc"] = df["report_clean"].map(lambda t: normalise_units(t))
df["report_expanded"] = df["report_proc"].map(expand_abbrev)
df["impression_proc"] = df["impression"].fillna("").map(normalise_units)
df["findings_proc"] = df["findings"].fillna("").map(normalise_units)
df["structured_text"] = [serialize_sections(s, i) for s, i in
                         zip(df["sections_json"], df["impression_proc"])]

print("Example structured serialisation:\n")
print(df["structured_text"].iloc[0][:600], "...")

Example structured serialisation:

[FINDINGS] [HEADER] ULTRASOUND EXAMINATION OF WHOLE ABDOMEN [CLINICAL INFORMATION] * Abdominal pain [LIVER] Normal in size with homogeneous echotexture. No focal or diffuse lesion seen. Portal vein, hepatic veins, Porta-hepatis & intra-hepatic portion of the IVC appear normal [GALL BLADDER] Normal in size and contour. Wall thickness is within normal limit. No evidence of echogenic structures or biliary sludge is seen within the gall bladder. Intrahepatic biliary radicals and common ducts are not dilated. Internaldiameter of CBD and that of portal vein are within normal limit [AORTA & IVC] Appe ...


## 7. Group-aware, label-stratified 5-fold split

`GroupKFold` respects groups but ignores labels; `StratifiedKFold` does the
reverse. Neither is sufficient here. The greedy assignment below handles both:
process groups largest-first, and place each into whichever fold currently has
the largest deficit on the labels that group carries.

Rare labels are weighted higher, because one misplaced positive out of three
matters more than one out of eighty.

In [10]:
rng = np.random.RandomState(CFG["seed"])
LABELS = multihot.columns.tolist()
Y = multihot.loc[df["study_id"]].values.astype(float)
K = CFG["n_folds"]

groups = df["dup_group"].values
uniq_groups = pd.Series(groups).value_counts()

group_y = {}
group_n = {}
for g in uniq_groups.index:
    idx = np.where(groups == g)[0]
    group_y[g] = Y[idx].sum(axis=0)
    group_n[g] = len(idx)

desired_lbl = np.tile(Y.sum(axis=0) / K, (K, 1))
desired_size = np.full(K, len(df) / K, dtype=float)
assign = {}

pending = list(uniq_groups.index)
pending.sort(key=lambda g: (-group_y[g].sum(), -group_n[g]))

for g in pending:
    gy, gn = group_y[g], group_n[g]
    active = np.where(gy > 0)[0]
    if len(active) == 0:
        cand = np.where(desired_size == desired_size.max())[0]
    else:
        remaining = Y.sum(axis=0) - sum(group_y[h] for h in assign)
        scarcity = np.where(remaining[active] > 0, remaining[active], np.inf)
        rarest = active[int(np.argmin(scarcity))]
        col = desired_lbl[:, rarest]
        cand = np.where(col == col.max())[0]
        if len(cand) > 1:
            sub = desired_size[cand]
            cand = cand[sub == sub.max()]
    k = int(rng.choice(cand))
    assign[g] = k
    desired_lbl[k] -= gy
    desired_size[k] -= gn

df["fold"] = df["dup_group"].map(assign)

fold_counts = np.zeros((K, len(LABELS)))
for k in range(K):
    fold_counts[k] = Y[df["fold"].values == k].sum(axis=0)

print(df["fold"].value_counts().sort_index())
print("\nfold size std:", round(float(df['fold'].value_counts().std()), 2),
      "(lower is better; a few studies of imbalance is normal with groups)")

fold
0    52
1    52
2    52
3    52
4    51
Name: count, dtype: int64

fold size std: 0.45 (lower is better; a few studies of imbalance is normal with groups)


In [11]:
dist = pd.DataFrame(fold_counts, columns=LABELS).astype(int)
dist.index.name = "fold"
show = dist[[l for l in LABELS if Y[:, LABELS.index(l)].sum() >= 5]]
print("Positives per fold (labels with ≥5 positives overall):")
print(show.to_string())

missing = [(k, l) for k in range(K) for l in show.columns if dist.loc[k, l] == 0]
if missing:
    print("\nLabels absent from a fold (expected for rarer classes):")
    for k, l in missing:
        print(f"  fold {k}: {l}")
else:
    print("\nEvery label with ≥5 positives appears in every fold.")

Positives per fold (labels with ≥5 positives overall):
      Bulky uterus  Cholelithiasis  Cystitis  Early pregnancy failure  Fatty liver  Hepatomegaly  Normal pregnancy  Normal study  Ovarian cyst  Pelvic collection  Renal cyst  Significant PVR  Thickened endometrium
fold                                                                                                                                                                                                                 
0                3               1         4                        3            6             3                 9            17             1                  2           1                2                      2
1                3               1         4                        3            5             3                 9            16             1                  1           1                2                      1
2                4               1         4                        3            5       

In [12]:
leak = 0
for k in range(K):
    tr = set(df.loc[df["fold"] != k, "dup_group"])
    te = set(df.loc[df["fold"] == k, "dup_group"])
    leak += len(tr & te)
print(f"duplicate groups straddling a train/test boundary: {leak}  (must be 0)")
assert leak == 0, "Split leaked — investigate before running any experiment."

duplicate groups straddling a train/test boundary: 0  (must be 0)


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(range(K), df["fold"].value_counts().sort_index().values, color="#2a6f97")
axes[0].set_xlabel("fold")
axes[0].set_ylabel("studies")
axes[0].set_title("Fold sizes")

top_lbl = [l for l in LABELS if Y[:, LABELS.index(l)].sum() >= 8]
norm = dist[top_lbl].div(dist[top_lbl].sum(axis=0), axis=1)
import seaborn as sns
sns.heatmap(norm.T, annot=True, fmt=".2f", cmap="RdYlGn", center=1 / K,
            vmin=0, vmax=2 / K, ax=axes[1], cbar_kws={"label": "share of positives"})
axes[1].set_title(f"Label balance across folds (ideal = {1/K:.2f})")
axes[1].set_xlabel("fold")
plt.tight_layout()
save("fig15_fold_balance")

## 8. Frame-level table and save

In [14]:
frames = frames.merge(
    df[["study_id", "fold", "dup_group", "primary_label", "primary_label_grouped",
        "exam_type"]],
    on="study_id", how="left")
frames.to_csv(ART / "frames.csv", index=False)

df.to_csv(ART / "master_split.csv", index=False)
multihot.to_csv(ART / "labels_multihot.csv")

split_card = {
    "n_folds": K,
    "fold_sizes": df["fold"].value_counts().sort_index().to_dict(),
    "n_groups": int(df["dup_group"].nunique()),
    "leaked_groups": int(leak),
    "n_frames": int(len(frames)),
    "frames_per_study_mean": round(float(per_study.mean()), 2),
    "img_size": CFG["img_size"],
}
json.dump(split_card, open(ART / "split_card.json", "w"), indent=2)
print(json.dumps(split_card, indent=2))

{
  "n_folds": 5,
  "fold_sizes": {
    "0": 52,
    "1": 52,
    "2": 52,
    "3": 52,
    "4": 51
  },
  "n_groups": 259,
  "leaked_groups": 0,
  "n_frames": 494,
  "frames_per_study_mean": 1.91,
  "img_size": 224
}


## Carry upstream artifacts forward

On Kaggle each notebook's output is its own dataset. Notebooks 03-06 need
`label_tiers.json` and `ontology.json`, which notebook **00** produced — if
this notebook's output does not contain them, the next notebook fails with a
`FileNotFoundError` and you have to attach two datasets instead of one.

Copying them here makes this output self-contained: attach only this one.

In [15]:
import shutil

REQUIRED_CARRY = ["ontology.json", "label_tiers.json", "labels_multihot.csv"]
OPTIONAL_CARRY = ["label_audit.csv", "label_frequency.csv", "dataset_card.json",
                  "master.csv", "eda_summary.json"]
for name in REQUIRED_CARRY + OPTIONAL_CARRY:
    dst = ART / name
    if dst.exists():
        continue
    src = None
    if (SRC / name).exists():
        src = SRC / name
    else:
        hits = sorted(Path("/kaggle/input").rglob(name))
        src = hits[0] if hits else None
    if src is not None:
        shutil.copy2(src, dst)
        print(f"  carried forward: {name}")
    elif name in REQUIRED_CARRY:
        print(f"  MISSING — attach notebook 00's output: {name}")
    else:
        print(f"  optional, not found (fine to skip): {name}")

required = ["master_split.csv", "frames.csv", "label_tiers.json",
            "labels_multihot.csv", "ontology.json"]
missing = [r for r in required if not (ART / r).exists()]
assert not missing, (
    f"This output is not self-contained; missing {missing}. Attach notebook 00's "
    "output to this notebook and re-run the cell above."
)
print("\nOutput is self-contained — notebooks 03-06 need only this dataset.")

  carried forward: ontology.json
  carried forward: label_tiers.json
  carried forward: label_audit.csv
  carried forward: label_frequency.csv
  carried forward: dataset_card.json
  carried forward: master.csv
  carried forward: eda_summary.json

Output is self-contained — notebooks 03-06 need only this dataset.


## Archive the frames

Panel decomposition produces ~860 small JPEGs spread over 259 subdirectories.
Kaggle's notebook-output datasets handle a single archive far more reliably
than hundreds of nested small files, and notebooks 03, 04 and 06 need the
images themselves. Writing `frames.zip` alongside the loose directory means
the downstream notebooks can recover the frames either way.

In [16]:
import zipfile

ZIP_PATH = Path("/kaggle/working/frames.zip")
_n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_STORED) as zf:
    for fp in sorted(FRAMES.rglob("*.jpg")):
        zf.write(fp, arcname=str(fp.relative_to(FRAMES)))   # e.g. R1/R1_p00.jpg
        _n += 1
print(f"Wrote {ZIP_PATH} — {_n} frames, {ZIP_PATH.stat().st_size/1e6:.1f} MB")

# Verify the archive round-trips before you rely on it.
with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
    assert len(names) == _n, "zip is incomplete"
    with zf.open(names[0]) as fh:
        _buf = np.frombuffer(fh.read(), np.uint8)
    _img = cv2.imdecode(_buf, cv2.IMREAD_COLOR)
    assert _img is not None and _img.std() > 1.0, "archived frames are unreadable"
print(f"Archive verified: {len(names)} entries, sample {_img.shape}, "
      f"pixel std {_img.std():.1f}")

Wrote /kaggle/working/frames.zip — 494 frames, 8.4 MB
Archive verified: 494 entries, sample (224, 224, 3), pixel std 99.3


In [17]:
print("Artifacts:")
for p in sorted(ART.iterdir()):
    if p.is_file():
        print(f"  {p.name:28s} {p.stat().st_size/1024:9.1f} KB")
print(f"\nFrames: {sum(1 for _ in FRAMES.rglob('*.jpg'))} files under {FRAMES}")

Artifacts:
  dataset_card.json                  0.5 KB
  eda_summary.json                   0.3 KB
  frames.csv                        57.3 KB
  label_audit.csv                   27.4 KB
  label_frequency.csv                1.1 KB
  label_tiers.json                   1.0 KB
  labels_multihot.csv               18.4 KB
  master.csv                      1274.6 KB
  master_split.csv                2261.2 KB
  ontology.json                      6.6 KB
  preprocess_config.json             0.2 KB
  split_card.json                    0.2 KB

Frames: 494 files under /kaggle/working/frames


---
### Save this as a Kaggle dataset

Commit the notebook, then **Output -> New Dataset**, including both
`artifacts/` and `frames/`. Name it something like
`ultrasound-fydp-processed` and attach it as the input to notebooks 03
onwards. Because of the carry-forward cell above it is the only dataset those
notebooks need.

Next: **03_baselines.ipynb**.